In [2]:
import os
import glob

import numpy as np

from scipy.interpolate import interp1d

import bagpipes as pipes

import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack

from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt

import sys

sys.path.append(os.path.abspath('..'))

from mrileyowens.dja import cutout
from mrileyowens.bagpipes import plot

In [18]:
def drop_old():

    home = os.getcwd()
    data = f'{home}/data'

    hdul_ryan = fits.open('JADES_f775w_dropouts_Endsley2024.fits')
    coords_ryan = SkyCoord(ra=hdul_ryan[1].data['RA'] * u.deg, dec=hdul_ryan[1].data['DEC'] * u.deg)

    max_sep = 0.1 * u.arcsec

    # Get the file paths of the JADES catalogs for GOODS-N and GOODS-S
    files = glob.glob(f'{data}/hlsp_jades_jwst_nircam_goods-*_photometry_v*.0_catalog.fits')

    # With the two catalogs opened
    with fits.open(files[0]) as hdul1:
        with fits.open(files[1]) as hdul2:

            # Open the data of the two catalogs as tables
            t1 = Table(hdul1['KRON_CONV'].data)
            t2 = Table(hdul2['KRON_CONV'].data)

            # Join the two catalogs together, using -1 as a fill value for any columns not shared by both catalogs
            t = vstack([t1,t2], join_type='outer')#, fill_value=-1)
            t = t.filled(np.nan)

            # Save the merged table
            t.write(f'{data}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits', format='fits', overwrite=True)

            coords_jades = SkyCoord(ra=t['RA'] * u.deg, dec=t['DEC'] * u.deg)

            #f775w_ABmag = (t['F775W_KRON_S'] * u.nJy).to(u.ABmag).value
            #f606w_ABmag = (t['F606W_KRON_S'] * u.nJy).to(u.ABmag).value

            #idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades)#[condition])

            #home = os.getcwd()
            #data = f'{home}/data'

            # Open the merged JADES GOODS-N and GOODS-S catalog
            #hdul = fits.open(f'{data}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits')
            #hdul = fits.open('merged.fits')

            # Open the merged catalog's data as a table
            #t = Table(hdul[1].data)

            # Set the F775W flux density to the 1 standard deviation upper limit if the SNR is < 1
            t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] / t['F775W_KRON_S_e'] < 1, t['F775W_KRON_S'] + 1 * t['F775W_KRON_S_e'], t['F775W_KRON_S'])

            # Sometimes the F775W flux density is slightly negative, despite the above operation. This complicates calculating a magnitude (it will be undefined), so assign
            # a small positive flux density (1e-3 nJy).
            t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] < 0, 1e-3, t['F775W_KRON_S'])

            # Calculate the F606W SNR. This is necessary to do before the below operation assigning upper limits to low SNR F606W photometry, which would alter the F606W
            # SNR if measured after that operation.
            f606w_snr = t['F606W_KRON_S'] / t['F606W_KRON_S_e']

            # Set the F606W flux density to the 1 standard deviation upper limit if the SNR is < 1
            t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] / t['F606W_KRON_S_e'] < 1, t['F606W_KRON_S'] + 1 * t['F606W_KRON_S_e'], t['F606W_KRON_S'])

            # Sometimes the F606W flux density is slightly negative, despite the above operation. This complicates calculating a magnitude (it will be undefined), so assign
            # a small positive flux density (1e-3 nJy).
            t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] < 0, 1e-3, t['F606W_KRON_S'])

            # Calculate AB magnitudes for photometry used to measure colors
            f606w_ABmag = (t['F606W_KRON_S'] * u.nJy).to(u.ABmag)
            f775w_ABmag = (t['F775W_KRON_S'] * u.nJy).to(u.ABmag)
            f090w_ABmag = (t['F090W_KRON_S'] * u.nJy).to(u.ABmag)
            f150w_ABmag = (t['F150W_KRON_S'] * u.nJy).to(u.ABmag)

            print(f775w_ABmag)
            print(f090w_ABmag)

            # Make boolean masks for the initial color selections. The first guarantees a strong break in F775W, corresponding to z ~ 6. The second ensures the spectrum
            # longward of the F775W break is much flatter, as would be expected for a dropout galaxy
            cond_color_1 = (f775w_ABmag - f090w_ABmag) > 1.2
            cond_color_2 = (f090w_ABmag - f150w_ABmag) < 1.0
            cond_color_3 = (f775w_ABmag - f090w_ABmag) > (f090w_ABmag - f150w_ABmag + 1.2 * u.mag)

            # Make a boolean mask for a low SNR in F435W. At z ~ 6, the Lyman break is completely longward of F435W, so this filter should have minimal flux for true
            # Lyman break galaxies.
            cond_lyc_snr = (t['F435W_KRON_S'] / t['F435W_KRON_S_e']) < 2

            # Make a boolean mask for a strong break in F606W, which z ~ 6 galaxies should show. If the SNR in F606W is low (< 2), relax the necessary break condition.
            cond_f606w_dropout_1 = (f606w_ABmag - f090w_ABmag) > 2.7
            cond_f606w_dropout_2 = (f606w_ABmag - f090w_ABmag) > 1.8
            cond_f606w_dropout = np.where(f606w_snr < 2, cond_f606w_dropout_2, cond_f606w_dropout_1)

            # Make a boolean mask for strong F775W breaks
            cond_f775w_dropout_strong = (f775w_ABmag - f090w_ABmag) > 2.5

            # Make a list of JWST filters in the joint JADES GOODS-N / GOODS-S catalog
            jwst_filters = ['F090W','F115W','F150W','F200W','F277W','F356W','F444W','F182M','F210M','F335M','F410M','F430M','F460M','F480M']

            # Make empty lists of the JWST photometry and uncertainties corresponding to the filters, to be filled in the loop below
            jwst_photometry = []
            jwst_photometry_e = []

            # For each JWST filter in the catalog
            for filter in jwst_filters:

                # Add the filter's photometry to the lists
                jwst_photometry.append([t[f'{filter}_KRON_S'].data])
                jwst_photometry_e.append([t[f'{filter}_KRON_S_e'].data])

            print(jwst_photometry)

            # Stack the JWST photometry and uncertainties
            jwst_photometry = np.vstack(jwst_photometry)
            jwst_photometry_e = np.vstack(jwst_photometry_e)

            print(jwst_photometry)

            # Calculate the SNR of the JWST photometry
            jwst_photometry_snr = jwst_photometry / jwst_photometry_e

            # Make a boolean mask for galaxies with any JWST filter with SNR > 5
            cond_snr_gtr_5 = np.any(jwst_photometry_snr > 5, axis=0)

            # Make a boolean mask for galaxies with at least 3 JWST filters with SNR > 3
            cond_3_snr_gtr_3 = np.sum(jwst_photometry_snr > 3, axis=0) >= 3

            # Make boolean masks for galaxies with F814W / F850LP SNRs > 3
            cond_f814w_snr_gtr_3 = (t['F814W_KRON_S'] / t['F814W_KRON_S_e']) > 3
            cond_f850lp_snr_gtr_3 = (t['F850LP_KRON_S'] / t['F850LP_KRON_S_e']) > 3

            #cond_f775w_e_nan = ~np.isnan(t['F775W_KRON_S_e'].data)

            cond = (cond_color_1 & cond_color_2 & cond_color_3 & ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong) & cond_snr_gtr_5 & cond_3_snr_gtr_3 & (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3))
            #cond = (cond_color_1 & cond_color_2 & cond_color_3 & ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong) & cond_snr_gtr_5 & cond_3_snr_gtr_3 & (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3))

            filtered = t[cond]
            print(type(filtered))
            print(len(filtered))

            #hdul_ryan = fits.open('JADES_f775w_dropouts_Endsley2024.fits')
            #coords_ryan = SkyCoord(ra=hdul_ryan[1].data['RA'] * u.deg, dec=hdul_ryan[1].data['DEC'] * u.deg)

            #hdul_jades = SkyCoord(ra=)
            #coords_jades = SkyCoord(ra=t['RA'] * u.deg, dec=t['DEC'] * u.deg)

            #max_sep = 0.1 * u.arcsec

            #for i, condition in enumerate([cond, cond_f606w_dropout_test, cond_f606w_dropout, cond_f606w_dropout_1, cond_f606w_dropout_2, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):
            #for i, condition in enumerate([cond, cond_f775w_e_nan, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):
            #for i, condition in enumerate([cond, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):
            for i, condition in enumerate([cond, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):

                #print(i)
                '''
                for coord in coords_ryan:
                    #idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades[condition])
                    idx, d2d, _ = coord.match_to_catalog_sky(coords_jades)#[condition])
                    #print(coords_ryan[d2d < max_sep])
                    print(f"Ryan: {coord}, JADES[{idx}]: {coords_jades[idx]}, d2d={d2d} {d2d.to(u.arcsec)}, passes={d2d < max_sep}")
                    if d2d < max_sep:
                        #matches = coords_jades[idx[d2d < max_sep]]
                        #print(f'{i}: ', len(matches), idx, d2d, coords_ryan[d2d < max_sep], matches)
                        print('pass!', coord, coords_jades[idx], coord.separation(coords_jades[idx]).to(u.arcsec))
                '''

                #idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades[condition])
                idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades)#[condition])
                #print(np.sum(np.isnan(f606w_ABmag[idx])), np.sum(t['F606W_KRON_S'][idx] < 0), np.sum(np.isnan(f775w_ABmag[idx])), np.sum(t['F775W_KRON_S'][idx] < 0))
                matches_in_condition = condition[idx]
                matches = coords_jades[idx[matches_in_condition & (d2d < max_sep)]]
                #ids = t['ID'][idx[~matches_in_condition & (d2d < max_sep)]]
                print(f'{i}: ', len(matches))#, ids)#, matches)

                # Checks for the F814W / F850LP SNR condition
                #print('F814W: ', (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F814W_e: ', (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F814W SNR: ', (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #f814w_snr = (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value

                #print('F814W SNR: ', np.nanmedian(f814w_snr), np.nanmean(f814w_snr), np.nanstd(f814w_snr))

                #print('F850LP: ', (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F850LP_e: ', (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F850LP SNR: ', (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #f850lp_snr = (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value

                #print('F850LP SNR: ', np.nanmedian(f850lp_snr), np.nanmean(f850lp_snr), np.nanstd(f850lp_snr))
                # ---

                # Checks for the F606W break or strong F775W break condition
                #print('F435W: ', (t['F435W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F435W_e: ', (t['F435W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F435W SNR: ', (t['F435W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F435W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #print('F606W: ', (t['F606W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F606W_e: ', (t['F606W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F090W_e: ', (t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #print('F606W - F090W: ', f606w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])

                #color_3 = f606w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
                #print('F606W - F090W: ', np.median(color_3[~np.isinf(color_3)]), np.mean(color_3[~np.isinf(color_3)]), np.std(color_3[~np.isinf(color_3)]))

                #print('F775W - F090W: ', f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])

                #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
                #print('F775W - F090W: ', np.nanmedian(color_1[~np.isinf(color_1)]), np.nanmean(color_1[~np.isinf(color_1)]), np.nanstd(color_1[~np.isinf(color_1)]))
                # ---

                # Checks for the F775W - F090W > F090W - F150W + 1.2 color condition
                #print('F775W: ', (t['F775W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F775W_e: ', (t['F775W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F090W_e: ', (t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F150W: ', (t['F150W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F150W_e: ', (t['F150W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F775W - F090W: ', f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])
                #print('F090W - F150W + 1.2: ', f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f150w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] + 1.2)

                #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
                #color_2 = f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f150w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]

                #check = color_1 - (color_2 + 1.2)

                #print('(F775W - F090W) - (F090W - F150W + 1.2): ', np.nanmedian(check[~np.isinf(check)]), np.nanmean(check[~np.isinf(check)]), np.nanstd(check[~np.isinf(check)]))
                # ---

                # Checks for the F775W - F090W color condition
                #print('F775W: ', (t['F775W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]

                #print('F775W - F090W: ', np.median(color_1[~np.isinf(color_1)]), np.mean(color_1[~np.isinf(color_1)]), np.std(color_1[~np.isinf(color_1)]))
                # ---

                #print('F090W: ', f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])
                #print((t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
                #print((t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

                #print(np.sum(f606w_snr[idx[matches_in_condition & (d2d < max_sep)]] < 2))

                ####
                #### Write this loop to plot the photometry and such of each galaxy of Ryan's that gets thrown out by a condition!
                ####

            tab = fits.BinTableHDU(filtered.as_array())

            hdul_new = fits.HDUList([fits.PrimaryHDU(), tab])
            hdul_new.writeto(f'{data}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits', overwrite=True)

def merge():

    '''
    Merge the GOODS-N and GOODS-S JADES photometric catalogs into a single catalog
    '''

    # Set common directories
    home = os.getcwd()
    data = f'{home}/data'
    results = f'{home}/results'

    # Get the file paths of the JADES catalogs for GOODS-N and GOODS-S
    files = glob.glob(f'{data}/hlsp_jades_jwst_nircam_goods-*_photometry_v*.0_catalog.fits')

    # With the two catalogs opened
    with fits.open(files[0]) as hdul1:
        with fits.open(files[1]) as hdul2:

            # -----------------------------------------------------
            # Construct a modifed PrimaryHDU for the merged catalog
            # -----------------------------------------------------

            # Make a new, blank PrimaryHDU
            hdu = fits.PrimaryHDU()

            # Open the primary HDUs of the constituent catalogs
            hdu1 = hdul1[0]
            hdu2 = hdul2[0]

            # For each key in the original HDUs
            for key in hdu1.header:

                # Combine the key values in the joined HDU if the two catalogs don't agree
                if hdu1.header[key] != hdu2.header[key]:
                    hdu.header[key] = f'{hdu1.header[key]},{hdu2.header[key]}'
                
                # Otherwise directly assign the original key value to the joined HDU
                else:
                    hdu.header[key] = hdu1.header[key]

            # --------------------------------------------------
            # Join together the HDUs of the constituent catalogs
            # --------------------------------------------------

            # Make a list of HDUs, to be appended later, starting with the joined PrimaryHDU
            hdul = [hdu]

            # List the extensions of the original catalogs to include in the joined catalog
            extensions = np.array(['FILTERS', 'FLAG', 'SIZE', 'KRON_CONV'], dtype=str)

            # For each of the original extensions to include in the joined catalog
            for extension in extensions:

                # Extract the extension's data as a table from the original catalogs
                t1 = Table(hdul1[extension].data)
                t2 = Table(hdul2[extension].data)

                # Stack the content of the two tables together
                t = vstack([t1,t2], join_type='outer')

                # Fill any missing values in the joined table with a -1
                t = t.filled(-1)

                # Convert the joiend table to a HDU
                hdu = fits.table_to_hdu(t)
                
                # Set the HDU's extension name
                hdu.header['EXTNAME'] = extension

                # Append the HDU to the HDU list
                hdul.append(hdu)

            # -------------------------------------------------
            # Save the joined HDUs into a single joined catalog
            # -------------------------------------------------

            # Formally convert the list of HDUs to a HDUL
            hdul = fits.HDUList(hdul)

            # Write the joined HDUL
            hdul.writeto(f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits', overwrite=True)

def drop():

    '''
    Apply the initial F775W dropout (z ~ 6) selections to the joint JADES GOODS-N and GOODS-S catalog
    '''

    # Set common directories
    home = os.getcwd()
    #data = f'{home}/data'
    results = f'{home}/results'

    # Open the merged JADES GOODS-N and GOODS-S catalog
    hdul = fits.open(f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog.fits')

    # Open the merged catalog's photometry as a table
    t = Table(hdul['KRON_CONV'].data)

    #coords_jades = SkyCoord(ra=t['RA'] * u.deg, dec=t['DEC'] * u.deg)

    #hdul_ryan = fits.open('JADES_f775w_dropouts_Endsley2024.fits')
    #coords_ryan = SkyCoord(ra=hdul_ryan[1].data['RA'] * u.deg, dec=hdul_ryan[1].data['DEC'] * u.deg)

    #max_sep = 0.1 * u.arcsec

    # ---------------------------------------------------------------------------------------------------------------------------------------
    # Set the photometry of the Lyman-break shortward filters used to calculate a dropout color to the 1-sigma upper limit when their SNR < 1
    # ---------------------------------------------------------------------------------------------------------------------------------------

    # Set the F775W flux density to the 1 standard deviation upper limit if the SNR is < 1
    #t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] / t['F775W_KRON_S_e'] < 1, t['F775W_KRON_S'] + 1 * t['F775W_KRON_S_e'], t['F775W_KRON_S'])
    t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] / t['F775W_KRON_S_e'] < 1, t['F775W_KRON_S_e'], t['F775W_KRON_S'])

    # Before adjusting the F606W photometry, calculate the F606W SNR. This is necessary to do before the next operation assigning upper limits 
    # to low SNR F606W photometry, which would alter the F606W SNR if measured after that operation.
    f606w_snr = t['F606W_KRON_S'] / t['F606W_KRON_S_e']

    # Set the F606W flux density to the 1 standard deviation upper limit if the SNR is < 1
    #t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] / t['F606W_KRON_S_e'] < 1, t['F606W_KRON_S'] + 1 * t['F606W_KRON_S_e'], t['F606W_KRON_S'])
    t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] / t['F606W_KRON_S_e'] < 1, t['F606W_KRON_S_e'], t['F606W_KRON_S'])

    # Sometimes the F606W and/or F775W flux density is slightly negative, despite the above operation. This complicates calculating a magnitude 
    # (it will be undefined), so assign a small positive flux density (1e-3 nJy).
    #t['F775W_KRON_S'] = np.where(t['F775W_KRON_S'] < 0, 1e-3, t['F775W_KRON_S'])
    #t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] < 0, 1e-3, t['F606W_KRON_S'])

    # Calculate the F606W SNR. This is necessary to do before the below operation assigning upper limits to low SNR F606W photometry, which would alter the F606W
    # SNR if measured after that operation.
    #f606w_snr = t['F606W_KRON_S'] / t['F606W_KRON_S_e']

    # Set the F606W flux density to the 1 standard deviation upper limit if the SNR is < 1
    #t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] / t['F606W_KRON_S_e'] < 1, t['F606W_KRON_S'] + 1 * t['F606W_KRON_S_e'], t['F606W_KRON_S'])

    # Sometimes the F606W flux density is slightly negative, despite the above operation. This complicates calculating a magnitude (it will be undefined), so assign
    # a small positive flux density (1e-3 nJy).
    #t['F606W_KRON_S'] = np.where(t['F606W_KRON_S'] < 0, 1e-3, t['F606W_KRON_S'])

    # ----------------------------------------------------------
    # Create boolean masks of the initial photometric selections
    # ----------------------------------------------------------

    # Calculate AB magnitudes for photometry used to measure colors. Use the unitless values since including units has been troublesome.
    f606w_ABmag = (t['F606W_KRON_S'] * u.nJy).to(u.ABmag).value
    f775w_ABmag = (t['F775W_KRON_S'] * u.nJy).to(u.ABmag).value
    f090w_ABmag = (t['F090W_KRON_S'] * u.nJy).to(u.ABmag).value
    f150w_ABmag = (t['F150W_KRON_S'] * u.nJy).to(u.ABmag).value

    # Make a boolean mask for a break in F775W
    cond_f775w_break = (f775w_ABmag - f090w_ABmag) > 1.2

    # Make a boolean mask for a much flatter near-IR 
    cond_flat = (f090w_ABmag - f150w_ABmag) < 1.0

    # Make a boolean mask for a much sharper F775W break than the observed near-IR slope
    cond_f775w_break_gtr = (f775w_ABmag - f090w_ABmag) > (f090w_ABmag - f150w_ABmag + 1.2)

    # Make a boolean mask for a low SNR in F435W. At z ~ 6, the Lyman break is completely longward of F435W, so this filter should have minimal 
    # flux for true Lyman break galaxies.
    cond_lyc_snr = (t['F435W_KRON_S'] / t['F435W_KRON_S_e']) < 2

    # Make a boolean mask for a strong break in F606W, which z ~ 6 galaxies should show. If the SNR in F606W is low (< 2), relax the necessary 
    # break condition.
    cond_f606w_break = (f606w_ABmag - f090w_ABmag) > 2.7
    cond_f606w_break_weak = (f606w_ABmag - f090w_ABmag) > 1.8
    cond_f606w_break = np.where(f606w_snr < 2, cond_f606w_break_weak, cond_f606w_break)

    # Make a boolean mask for strong F775W breaks
    cond_f775w_break_strong = (f775w_ABmag - f090w_ABmag) > 2.5

    # ------------------------------------------------------
    # Create boolean masks of the photometric SNR selections
    # ------------------------------------------------------

    # Make a list of JWST filters in the joint JADES GOODS-N / GOODS-S catalog
    jwst_filters = ['F090W','F115W','F150W','F200W','F277W','F335M','F356W','F410M','F444W']

    # Make empty lists of the JWST photometry and uncertainties corresponding to the filters, to be filled in the loop below
    jwst_photometry = []
    jwst_photometry_e = []

    # For each JWST filter in the catalog
    for filter in jwst_filters:

        # Add the filter's photometry to the lists
        jwst_photometry.append([t[f'{filter}_KRON_S'].data])
        jwst_photometry_e.append([t[f'{filter}_KRON_S_e'].data])

    # Stack the JWST photometry and uncertainties
    jwst_photometry = np.vstack(jwst_photometry)
    jwst_photometry_e = np.vstack(jwst_photometry_e)

    # Calculate the SNR of the JWST photometry
    jwst_photometry_snr = jwst_photometry / jwst_photometry_e

    # Make a boolean mask for objects with any JWST filter with SNR > 5
    cond_snr_gtr_5 = np.any(jwst_photometry_snr > 5, axis=0)

    # Make a boolean mask for objects with at least 3 JWST filters with SNR > 3
    cond_3_snr_gtr_3 = np.sum(jwst_photometry_snr > 3, axis=0) >= 3

    # Make boolean masks for objects with F814W / F850LP SNRs > 3
    cond_f814w_snr_gtr_3 = (t['F814W_KRON_S'] / t['F814W_KRON_S_e']) > 3
    cond_f850lp_snr_gtr_3 = (t['F850LP_KRON_S'] / t['F850LP_KRON_S_e']) > 3

    #cond_real_errors = ~np.isnan(t['F435W_KRON_S_e']) & ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])
    cond_real_errors = ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])

    # ---------------------------------------------------
    # Create boolean masks for artifacts or contamination
    # ---------------------------------------------------

    cond_real_errors = ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])

    # Make a boolean mask of non-stars
    cond_star = Table(hdul['FLAG'].data)['FLAG_ST'] != 1

    # Make a boolean mask of objects not contaminated by bright stars
    cond_bs = Table(hdul['FLAG'].data)['FLAG_BS'] != 1.0

    # Make a boolean mask of objects that don't have a bright neighbor >10x as bright
    cond_bn = Table(hdul['FLAG'].data)['FLAG_BN'] != 2.0

    # ----------------------------------------------
    # Combine and apply the individual boolean masks
    # ----------------------------------------------

    # Combine all the boolean masks into a single mask
    conditions = (cond_star & cond_bs & cond_bn & cond_real_errors & cond_f775w_break & cond_flat & cond_f775w_break_gtr & ((cond_lyc_snr & cond_f606w_break) | cond_f775w_break_strong) & cond_snr_gtr_5 & cond_3_snr_gtr_3 & (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3))

    # Mask the table
    t = t[conditions]
    print(len(t))

    # Make a HDU from the table
    tab = fits.BinTableHDU(t.as_array())

    # Make a HDUL from the table and save it
    hdul = fits.HDUList([fits.PrimaryHDU(), tab])
    hdul.writeto(f'{results}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits', overwrite=True)

    #for i, condition in enumerate([conditions, cond_real_errors, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):

    #for i, condition in enumerate([cond, cond_f606w_dropout_test, cond_f606w_dropout, cond_f606w_dropout_1, cond_f606w_dropout_2, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):
    #for i, condition in enumerate([cond, cond_f775w_e_nan, cond_color_1, cond_color_2, cond_color_3, ((cond_lyc_snr & cond_f606w_dropout) | cond_f775w_dropout_strong), cond_snr_gtr_5, cond_3_snr_gtr_3, (cond_f814w_snr_gtr_3 | cond_f850lp_snr_gtr_3)]):
    for i, condition in enumerate([conditions, ((cond_lyc_snr & cond_f606w_break) | cond_f775w_break_strong)]):

        pass

        #print(i)
        '''
        for coord in coords_ryan:
            #idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades[condition])
            idx, d2d, _ = coord.match_to_catalog_sky(coords_jades)#[condition])
            #print(coords_ryan[d2d < max_sep])
            print(f"Ryan: {coord}, JADES[{idx}]: {coords_jades[idx]}, d2d={d2d} {d2d.to(u.arcsec)}, passes={d2d < max_sep}")
            if d2d < max_sep:
                #matches = coords_jades[idx[d2d < max_sep]]
                #print(f'{i}: ', len(matches), idx, d2d, coords_ryan[d2d < max_sep], matches)
                print('pass!', coord, coords_jades[idx], coord.separation(coords_jades[idx]).to(u.arcsec))
        '''

        '''
        #idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades[condition])
        idx, d2d, _ = coords_ryan.match_to_catalog_sky(coords_jades)#[condition])
        matches_in_condition = condition[idx]
        matches = coords_jades[idx[matches_in_condition & (d2d < max_sep)]]
        #ids = t['ID'][idx[~matches_in_condition & (d2d < max_sep)]]
        print(f'{i}: ', len(matches))#, ids)#, matches)
        '''

        # Checks for the F814W / F850LP SNR condition
        #print('F814W: ', (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F814W_e: ', (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F814W SNR: ', (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #f814w_snr = (t['F814W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F814W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value

        #print('F814W SNR: ', np.nanmedian(f814w_snr), np.nanmean(f814w_snr), np.nanstd(f814w_snr))

        #print('F850LP: ', (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F850LP_e: ', (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F850LP SNR: ', (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #f850lp_snr = (t['F850LP_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F850LP_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value

        #print('F850LP SNR: ', np.nanmedian(f850lp_snr), np.nanmean(f850lp_snr), np.nanstd(f850lp_snr))
        # ---

        # Checks for the F606W break or strong F775W break condition
        #print('F435W: ', (t['F435W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F435W_e: ', (t['F435W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F435W SNR: ', (t['F435W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value / (t['F435W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #print('F606W: ', (t['F606W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F606W_e: ', (t['F606W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F090W_e: ', (t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #print('F606W - F090W: ', f606w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])

        #color_3 = f606w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
        #print('F606W - F090W: ', np.median(color_3[~np.isinf(color_3)]), np.mean(color_3[~np.isinf(color_3)]), np.std(color_3[~np.isinf(color_3)]))

        #print('F775W - F090W: ', f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])

        #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
        #print('F775W - F090W: ', np.nanmedian(color_1[~np.isinf(color_1)]), np.nanmean(color_1[~np.isinf(color_1)]), np.nanstd(color_1[~np.isinf(color_1)]))
        # ---

        # Checks for the F775W - F090W > F090W - F150W + 1.2 color condition
        #print('F775W: ', (t['F775W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F775W_e: ', (t['F775W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F090W_e: ', (t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F150W: ', (t['F150W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F150W_e: ', (t['F150W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F775W - F090W: ', f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])
        #print('F090W - F150W + 1.2: ', f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f150w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] + 1.2)

        #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]
        #color_2 = f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f150w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]

        #check = color_1 - (color_2 + 1.2)

        #print('(F775W - F090W) - (F090W - F150W + 1.2): ', np.nanmedian(check[~np.isinf(check)]), np.nanmean(check[~np.isinf(check)]), np.nanstd(check[~np.isinf(check)]))
        # ---

        # Checks for the F775W - F090W color condition
        #print('F775W: ', (t['F775W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print('F090W: ', (t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #color_1 = f775w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]] - f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]]

        #print('F775W - F090W: ', np.median(color_1[~np.isinf(color_1)]), np.mean(color_1[~np.isinf(color_1)]), np.std(color_1[~np.isinf(color_1)]))
        # ---

        #print('F090W: ', f090w_ABmag[idx[~matches_in_condition & (d2d < max_sep)]])
        #print((t['F090W_KRON_S'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)
        #print((t['F090W_KRON_S_e'][idx[~matches_in_condition & (d2d < max_sep)]] * u.nJy).value)

        #print(np.sum(f606w_snr[idx[matches_in_condition & (d2d < max_sep)]] < 2))
    #ids = [6530, 6721, 6990, 8706, 31077, 31538, 34773, 36591, 46089, 46311, 57118]

    '''
    for id in ids:

        idx = np.where(t['ID'] == id)

        print(t[idx].pprint_all())

        for col in ['F435W_KRON_S', 'F606W_KRON_S', 'F775W_KRON_S', 'F814W_KRON_S', 'F850LP_KRON_S', 'F090W_KRON_S', 'F150W_KRON_S']:

            print(col, t[col].data[idx], t[f'{col}_e'].data[idx])
    '''

    # Make a HDU from the table
    #tab = fits.BinTableHDU(filtered.as_array())

    ## Make a HDUL from the table and save it
    #hdul = fits.HDUList([fits.PrimaryHDU(), tab])
    #hdul.writeto(f'{results}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits', overwrite=True)

def cutouts():

    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'

    hdul = fits.open(f'{data}/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits')

    ids = hdul[1].data['ID']
    ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

    for i, _ in enumerate(ra):

        cutout(ra[i], dec[i], filters=['f435w', 'f606w', 'f775w','f814w','f850lp','f090w-clear','f150w-clear'], id=ids[i], save=True, dir='figs/f775w_dropouts_init_cutouts')

def drop_artifacts():

    # ?
    #ids = [139844]
    # Brown dwarfs?
    #bds = [55042, 59824, 149678, 208953, 1014389, 1021032, 1036685, 1088892]
    # Bright, point-like objects, diffraction spikes, and other
    '''
    other = [161614, 169404, 177745, 189201, 207204, 251817, 284377, 285104, 295724, 1027608, 1027862, 1062954, 1073055,
        1078196, 1078197, 1080770, 1080833, 1120433, 1120434, 1120440, 1120444, 1121860, 1156517, 1156519, 1177583,
        1177584, 1177585, 1178028, 1178046, 1178054, 1178091, 1178120, 1178425, 1178430, 1178441, 1178442, 1181020,
        1181026, 1181057, 1181103, 1181107, 1181112, 1181114, 1181123, 1181139, 1181148, 1181223, 1181224, 1182785,
        1182798, 1182799, 1183744, 1183791, 1183875, 1183997, 1184278]
    '''

    diffraction_spikes = [251817, 1156517, 1156519, 1177585, 1183997]

def fit():

    '''
    Make the final selection on the F775W dropout candidates using Bagpipes fits
    '''

    def load_id(id):

        '''
        Load the photometry of an object from the merged JADES catalog of F775W dropout candidates given its ID in a Bagpipes-friendly format
        '''

        #catalog = 'results/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits'

        #hdul = fits.open(catalog)

        #cols = hdul[1].data.columns.names

        # Get the index of the object's ID in the merged catalog
        idx = np.where(ids == int(id))[0][0]

        # Extract the photometry at the object's index, in uJy
        phot_mujy = np.array([hdul[1].data[f'{col}_KRON_S'][idx] / 1e3 for col in filters_jades])
        phot_err_mujy = np.array([hdul[1].data[f'{col}_KRON_S_e'][idx] / 1e3 for col in filters_jades])

        # Make a boolean mask to select photometric artifacts
        mask = np.isnan(phot_mujy) | np.isinf(phot_mujy) | np.isnan(phot_err_mujy) | np.isinf(phot_err_mujy) | (phot_err_mujy < 0) | (phot_mujy < 0)

        # Replace photometric artifacts with dummy values that simulate an unconstrained data point
        phot_mujy = np.where(mask, 1e-10, phot_mujy)
        phot_err_mujy = np.where(mask, 1e10, phot_err_mujy)

        # Stack the observed photometry and uncertainties together
        photometry = np.c_[phot_mujy, phot_err_mujy]

        return photometry

    #f_fuv = 
    #filters = ['JWST_NIRCam.F090W', 'JWST_NIRCam.F115W', 'JWST_NIRCam.F150W', 'JWST_NIRCam.F182M', 'JWST_NIRCam.F200W', 'JWST_NIRCam.F210M', 'JWST_NIRCam.F277W', 'JWST_NIRCam.F335M', 'JWST_NIRCam.F356W', 'JWST_NIRCam.F410M', 'JWST_NIRCam.F430M', 'JWST_NIRCam.F444W', 'JWST_NIRCam.F460M', 'JWST_NIRCam.F480M', 
    #    'HST_ACS_WFC.F435W', 'HST_ACS_WFC.F606W', 'HST_ACS_WFC.F775W', 'HST_ACS_WFC.F814W', 'HST_ACS_WFC.F850LP', 'HST_WFC3_IR.F105W', 'HST_WFC3_IR.F125W', 'HST_WFC3_IR.F140W', 'HST_WFC3_IR.F160W']
    
    ## Create a list of file paths to filter curves for the photometry of the initial dropout selections
    #filters = ['HST_ACS_WFC.F435W', 'HST_ACS_WFC.F606W', 'HST_ACS_WFC.F775W', 'HST_ACS_WFC.F814W', 'HST_ACS_WFC.F850LP',
    #'JWST_NIRCam.F090W', 'JWST_NIRCam.F115W', 'JWST_NIRCam.F150W', 'JWST_NIRCam.F200W', 'JWST_NIRCam.F277W', 'JWST_NIRCam.F335M', 'JWST_NIRCam.F356W', 'JWST_NIRCam.F410M', 'JWST_NIRCam.F444W']
    #filt_list = [f'../mrileyowens/filters/bagpipes/{filter}.dat' for filter in filters]

    #filters_jades = [filter.split('.')[1] for filter in filters]

    # Set common directories
    home = os.getcwd()
    results = f'{home}/results'

    # Set the file path to the initial F775W dropout candidates from the merged JADES catalog
    catalog = f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits'

    # Open the initial F775W dropout candidates catalog
    hdul = fits.open(catalog)

    # Open the photometry of the initial F775W dropout candidates as a table
    t = Table(hdul[1].data)

    # Get the object IDs of the initial F775W dropout candidates
    ids = t['ID']

    # ------------------------------------------
    # Assemble the Bagpipes fitting instructions
    # ------------------------------------------

    # Set the properties of the CSFH
    constant = {}
    constant['age_max'] = (0.,0.5)
    constant['age_min'] = (0.,0.5)
    constant['massformed'] = (6.,13.)
    constant['metallicity'] = (0.,0.5)

    # Set the nebular ionization parameter
    nebular = {}
    nebular['logU'] = -2

    # Set the dust properties
    dust = {}
    dust['type'] = 'Calzetti'
    dust['Av'] = (0.,2.)

    # Assemble the individual instructions and redshift constraints into a single dictionary
    instructions = {}
    instructions['redshift'] = (5.,7.)
    instructions['constant'] = constant
    instructions['nebular'] = nebular
    instructions['dust'] = dust

    # -----------------------------------------------
    # Make the Bagpipes filter list of the photometry
    # -----------------------------------------------

    # Create a list of file paths to filter curves for the photometry to include in the Bagpipes fits
    filters = ['HST_ACS_WFC.F435W', 'HST_ACS_WFC.F606W', 'HST_ACS_WFC.F775W', 'HST_ACS_WFC.F814W', 'HST_ACS_WFC.F850LP',
        'JWST_NIRCam.F090W', 'JWST_NIRCam.F115W', 'JWST_NIRCam.F150W', 'JWST_NIRCam.F200W', 'JWST_NIRCam.F277W', 'JWST_NIRCam.F335M', 'JWST_NIRCam.F356W', 'JWST_NIRCam.F410M', 'JWST_NIRCam.F444W']
    filt_list = [f'../mrileyowens/filters/bagpipes/{filter}.dat' for filter in filters]

    # Make a list of the same filters in the JADES catalog, matching the format of the table columns used in the merged catalog
    filters_jades = [filter.split('.')[1] for filter in filters]

    # ----------------------------------------------------------------------------------------------------------------
    # Fit the initial F775W dropout candidates with Bagpipes to measure their rest-UV flux densities at 1500 angstroms
    # ----------------------------------------------------------------------------------------------------------------

    # Make an empty list of the sampled flux density at rest-frame 1500 angstroms, to be appended in the loop below
    f_1500s = []

    # For each object ID
    for id in ids:

        # Instantiate the object and its photometry
        galaxy = pipes.galaxy(id, load_id, spectrum_exists=False, filt_list=filt_list, phot_units='mujy')

        # Fit the object with the given fitting instructions
        fit = pipes.fit(galaxy, instructions, run=catalog.split('/')[-1])
        fit.fit(verbose=False, sampler='nautilus')

        # Also get advanced quantities from the fitted parameters
        fit.posterior.get_advanced_quantities()

        # Get the rest wavelengths of the model SEDs
        w_rest_angstrom = fit.posterior.model_galaxy.wavelengths * u.angstrom

        # Get the rest-frame wavelength-space flux densities of the model SEDs
        seds_rest_ergscma = fit.posterior.samples['spectrum_full'] * u.erg / u.s / u.cm**2 / u.angstrom

        # Get the best-fit redshifts
        zs = fit.posterior.samples['redshift']

        # Convert the rest-frame wavelengths to observed-frame wavelengths
        w_obs_angstrom = w_rest_angstrom * (1 + zs)[:,None]

        # Conver the rest-frame wavelength-space flux densities to observed-frame frequency-space flux densities
        seds_obs_njy = seds_rest_ergscma.to(u.nJy, u.spectral_density(w_obs_angstrom))

        # Plot the Bagpipes fit
        plot([id], load_id, instructions, filt_list, save=True, dir=f'figs/bagpipes_fits/', run=catalog.split('/')[-1])

        # Make an empty list of the flux densities measured at rest-frame 1500 angstroms from the model SEDs in the posterior, to be appended below
        f_1500 = []

        # For each model SED in the posterior
        for i, sed in enumerate(seds_obs_njy):

            # Interpolate the SED
            f = interp1d(w_obs_angstrom[i].value, sed.value, kind='cubic', bounds_error=False, fill_value=np.nan)    
            #sed_obs_intp_njy = f(w_rest_angstrom.value)

            # Append the flux density of the SED at the observed-frame wavelength of rest-frame 1500 angstroms
            f_1500.append(f(1500 * (1 + zs[i])))

        # Calculate the median of the flux densities measured from the model SEDs
        f_1500 = np.nanmedian(f_1500)

        # Append the median flux density to the master list of the flux density measured for each object
        f_1500s.append(f_1500)

    # Add units to the measured flux densities
    f_1500s = f_1500s * u.nJy

    # Create a boolean mask where the measured flux density is at least >3x the uncertainty in the four chosen longer-wavelength filters
    mask = ((f_1500s / (t['F335M_KRON_e'] * u.nJy)) > 3) \
        & ((f_1500s / (t['F356W_KRON_e'] * u.nJy)) > 3) \
        & ((f_1500s / (t['F410M_KRON_e'] * u.nJy)) > 3) \
        & ((f_1500s / (t['F444W_KRON_e'] * u.nJy)) > 3)

    # Mask the table
    filtered = t[mask]

    print(len(filtered))
    print(filtered.colnames)
    print(filtered)
    #print(type(filtered['ID'].data[0]))

    # Make a HDU from the table
    table = fits.BinTableHDU(filtered.as_array())

    # Make a HDUL from the table and save it
    hdul = fits.HDUList([fits.PrimaryHDU(), table])
    hdul.writeto(f'{results}/catalogs/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_final.fits', overwrite=True)

def other():

    # For each object
    for row in filtered:

        # Get the coordinates of the object
        id = row['ID']
        ra = row['RA']
        dec = row['DEC']

        cutout(ra, dec, id, save=True, dir='results/sel/cutouts')

    # IDs of objects that are diffraction spikes
    remove = [172786,176038,177146,179371,207204,284473,1062954,1068699,1102448,1078197,
        1078196,1073055,1120440,1120434,1120433,1120444,1178061,1178054,1178047,1178028,
        1178065,1178091,1178120,1178441,1181114,1181112,1181107,1181103,1181057,1181123,
        1181139,1181224,1182797,1182798,1182799,1183744,1183791,1184278]

    mask = ~np.isin(filtered['ID'], remove)
    filtered = filtered[mask]

    print(len(filtered))

    #print(type(filtered['ID']))

    '''
    # Get the number of rows (objects) in either catalog
    nrows1 = hdul1['KRON_CONV'].data.shape[0]
    nrows2 = hdul2['KRON_CONV'].data.shape[0]
            
    # Calculate the total number of rows in the final table
    nrows = nrows1 + nrows2

    # Make a new HDU from the columns of the first catalog, with the total number of rows
    hdu = fits.BinTableHDU.from_columns(hdul1['KRON_CONV'].columns, nrows=nrows)

    # For each column in the first catalog
    for i, colname in enumerate(hdul1['KRON_CONV'].columns.names):
                
        # If the column also exists in the second catalog
        if colname in hdul2['KRON_CONV'].columns.names:

            # Add the column values from the second catalog
            hdu.data[colname][nrows1:] = hdul2['KRON_CONV'].data[colname]

        else:

            print(colname)

            # Add dummy values to the table
            hdu.data[colname][nrows1:] = -1
    '''

In [ ]:
drop_old()

In [4]:
merge()

In [ ]:
drop()

/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: divide by zero encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/astropy/units/function/logarithmic.py:67: RuntimeWarning: invalid value encountered in log10
  return dex.to(self._function_unit, np.log10(x))
/var/folders/md/b3rxd7_x6zgd55s__25hxcwh0000gn/T/ipykernel_9147/755965781.py:401: RuntimeWarning: invalid value encountered in subtract
  cond_flat = (f090w_ABmag - f150w_ABmag) < 1.0
/var/folders/md/b3rxd7_x6zgd55s__25hxcwh0000gn/T/ipykernel_9147/755965781.py:404: RuntimeWarning: invalid value encountered in subtract
  cond_f775w_break_gtr = (f775w_ABmag - f090w_ABmag) > (f090w_ABmag - f150w_ABmag + 1.2)


698


In [ ]:
cutouts()

In [ ]:
catalog = 'data/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits'
hdul = fits.open(catalog)
print(hdul[1].data.columns.names)
print(hdul[1].data['F090W_KRON_S'])

In [ ]:
fit()


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/698.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/698.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/698.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/698.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/1675.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_good

/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/bagpipes/models/star_formation_history.py:131: RuntimeWarning: divide by zero encountered in log10
  self.ssfr = np.log10(self.sfr) - self.stellar_mass
/Users/a15136/Documents/research/.venv/lib/python3.13/site-packages/bagpipes/models/star_formation_history.py:132: RuntimeWarning: divide by zero encountered in log10
  self.nsfr = np.log10(self.sfr*self.age_of_universe) - self.formed_mass



Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/8706.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/8706.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/8892.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/8892.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/8892.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_

/Users/a15136/Documents/research/mrileyowens/bagpipes.py:25: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots()



Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/23699.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/23699.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/24388.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/24388.h5. To start over delete this file or change run.


Results loaded from pipes/posterior/hlsp_jades_jwst_nircam_goods-n_v1.0_goods-s-deep_v2.0_photometry_catalog_f775w_dropouts_init.fits/24388.h5

Fitting not performed as results have already been loaded from pipes/posterior/hlsp_jades_jwst_ni